# RS-385 load ramp-up: signal curves (Phase 6.5)

Plots the f(t) signals from the headless harness (`data/phase6-signals.csv`): the load ramp, speed, current, and the rotor-imbalance vibration (waveform, spectrum, waterfall, startup ring-down).

Data: RS-385 @ 7 V, one Load Torque ramping 2 to 22 mN·m over 10 s; rotor imbalance fault; sampled at 1 kHz (decimated from the 5 kHz sim). The first 0.5 s (housing ring-up transient) is dropped from the steady analysis but plotted on its own at the end.

Throughout, "1x" means once per shaft revolution (i.e. at the rotation frequency): the imbalance heavy spot passes a fixed point once per turn, so its signature lands at 1x.

Run all cells. Needs only `numpy`, `pandas`, `matplotlib`.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

df = pd.read_csv('data/phase6-signals.csv')
df = df[df.t >= 0.5].reset_index(drop=True)   # drop the startup ring-up
fs = 1.0 / np.median(np.diff(df.t))            # sample rate [Hz]
print(f'{len(df)} samples, fs approx {fs:.0f} Hz, t = {df.t.min():.2f} to {df.t.max():.2f} s')
df.head()

## 1. Load tau(t) and speed omega(t): the cause and the first response

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(df.t, df.tau * 1e3, color='tab:red', label='tau_load')
ax1.set_xlabel('t [s]'); ax1.set_ylabel('tau_load [mN·m]', color='tab:red'); ax1.tick_params(axis='y', labelcolor='tab:red')
ax2 = ax1.twinx()
ax2.plot(df.t, df.omega, color='tab:blue', label='omega')
ax2.set_ylabel('omega [rad/s]', color='tab:blue'); ax2.tick_params(axis='y', labelcolor='tab:blue')
plt.title('Load ramps up, speed falls (quasi-steady: tau_m approx 10 ms much less than the ramp)')
fig.tight_layout(); plt.show()

## 2. Armature current I(t): the mirror of omega

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.plot(df.t, df.current, color='tab:green')
plt.xlabel('t [s]'); plt.ylabel('I [A]'); plt.title('As the load rises (and speed falls), armature current rises with it: I approx tau/Kt')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 3. Radial vibration accY(t): a down-chirp in a decaying envelope

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.plot(df.t, df.accelY, lw=0.4, color='tab:purple')
plt.xlabel('t [s]'); plt.ylabel('accY [m/s²]'); plt.title('Imbalance 1x vibration: amplitude shrinks AND frequency slides down')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 4. accY zoomed: the actual sinusoid, early (fast/big) vs late (slow/small)

In [ ]:
def window(tc, dur=0.08):
    m = (df.t >= tc) & (df.t < tc + dur)
    return df.t[m].values, df.accelY[m].values
fig, axs = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, tc, lbl in zip(axs, [1.0, 10.0], ['early (t approx 1 s, omega high)', 'late (t approx 10 s, omega low)']):
    tt, yy = window(tc)
    ax.plot((tt - tc) * 1e3, yy, color='tab:purple')
    ax.set_xlabel('t - t0 [ms]'); ax.set_title(lbl); ax.grid(alpha=0.3)
axs[0].set_ylabel('accY [m/s²]')
fig.suptitle('Same 1x tone, lower frequency + smaller amplitude as the load grows'); fig.tight_layout(); plt.show()

## 5. Spectrum of accY: one 1x peak, sliding down

In [ ]:
def spectrum(tc, dur=1.0):
    m = (df.t >= tc) & (df.t < tc + dur)
    y = df.accelY[m].values; y = y - y.mean()
    n = len(y); w = np.hanning(n)
    Y = np.abs(np.fft.rfft(y * w)) * 2 / (w.sum())
    fr = np.fft.rfftfreq(n, 1 / fs)
    return fr, Y
plt.figure(figsize=(9, 3.5))
for tc, c in [(1.0, 'tab:blue'), (5.0, 'tab:orange'), (10.0, 'tab:green')]:
    fr, Y = spectrum(tc); plt.plot(fr, Y, color=c, label=f't approx {tc:.0f} s')
plt.xlim(0, 200); plt.xlabel('frequency [Hz]'); plt.ylabel('|accY| [m/s²]')
plt.title('1x peak walks DOWN (122 to 71 Hz) and shrinks as the load ramps'); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 6. Waterfall (spectrogram): the 1x ridge drifting down-left

In [ ]:
y = (df.accelY - df.accelY.mean()).values
plt.figure(figsize=(9, 4))
plt.specgram(y, NFFT=512, Fs=fs, noverlap=256, cmap='magma')
plt.ylim(0, 200); plt.xlabel('t [s] (from 0.5 s)'); plt.ylabel('frequency [Hz]')
plt.title('Imbalance 1x ridge sliding down as the motor is loaded'); plt.colorbar(label='power [dB]'); plt.tight_layout(); plt.show()

## 7. Measured 1x vs the omega^2 and omega^4 laws

The imbalance FORCE is proportional to omega^2 (centrifugal), but an accelerometer reads acceleration = omega^2 times displacement, and the displacement itself follows the force (proportional to omega^2) well below the 500 Hz bracket mode. So the ideal accelerometer limit is omega^4. The measured 1x sits BETWEEN omega^2 and omega^4 (here about omega^3.3): steeper than the force because of the double differentiation, but flatter than omega^4 because the discrete 4 kHz bracket integrator adds damping at higher drive frequency. It is NOT a resonance effect (the resonance is far above the band).

In [ ]:
secs = np.arange(int(df.t.min()) + 1, int(df.t.max()) + 1)
amp, om = [], []
for s in secs:
    m = (df.t >= s) & (df.t < s + 1)
    a = df.accelY[m].values; amp.append(np.sqrt(2) * np.std(a)); om.append(df.omega[m].mean())
amp = np.array(amp); om = np.array(om)
omega2 = amp[0] * (om / om[0]) ** 2   # pure omega^2 force law, anchored at the first point
omega4 = amp[0] * (om / om[0]) ** 4   # ideal accelerometer limit (omega^2 of an omega^2 displacement)
p = np.polyfit(np.log(om), np.log(amp), 1)[0]   # measured power
plt.figure(figsize=(9, 3.8))
plt.plot(secs, amp, 'o-', color='tab:purple', label=f'measured 1x (sqrt(2)*RMS), ~omega^{p:.1f}')
plt.plot(secs, omega2, '--', color='gray', label='omega^2 force law')
plt.plot(secs, omega4, ':', color='tab:red', label='ideal omega^4 (accel of displacement)')
plt.xlabel('t [s]'); plt.ylabel('1x amplitude [m/s²]')
plt.title('Measured 1x sits between the omega^2 force law and the ideal omega^4'); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 8. Startup ring-down (why the first 0.5 s is skipped)

At t = 0 the rotor already spins at full speed, so the imbalance force hits the accelerometer bracket (at rest) as a near-step. The acceleration overshoots to about 30 m/s² (about 9x the steady ~3 m/s² 1x), then rings down at the bracket's own ~500 Hz natural frequency. The ideal decay constant is 1/(zeta*omega_n) approx 16 ms, but the implicit-Euler integrator adds numerical damping, so as simulated it settles (~4 tau, tau approx 7 ms) by about 30 ms. The per-second decomposition starts at 0.5 s, long after this has died out.

In [ ]:
sd = pd.read_csv('data/phase6-startup.csv')   # full-rate (5 kHz) housing accelY at startup
steady = np.sqrt(2) * sd.accelY[sd.t >= 0.08].std()
plt.figure(figsize=(9, 3.6))
plt.plot(sd.t * 1e3, sd.accelY, color='tab:purple', lw=0.7)
plt.axhspan(-steady, steady, color='tab:purple', alpha=0.12, label=f'settled 1x +/-{steady:.1f} m/s²')
plt.axvline(30, color='tab:green', ls='--', label='settled ~30 ms (4*tau, tau approx 7 ms)')
plt.xlabel('time [ms]'); plt.ylabel('accY [m/s²]')
plt.title('Startup ring-down: ~500 Hz natural mode, overshoot ~30 m/s², damped by ~30 ms'); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()